# ENEL 441 lab 4 - PD control

# PD Control

PD control has 2 actions:
- proportional action
- derivative action

## Intuitition of Derivative action 

The PD controller is defined as:
$$
u(t) = K_p\varepsilon(t) + K_d \frac{d\varepsilon(t)}{dt}, 
$$
where $u$ is the output of the controller, $\varepsilon$ is the error $r(t) - y(t)$. 

Derivative action is proportional to the rate of change of the error. If the error is rapidly decreasing the derivative controller will act to slow down the controller. For example, if $K_p \varepsilon(t)$ is very big and quickly decreasing, i.e. the system is rapidly approaching the desired output. The output of the derivative term is negative, so it will remove part of the control action, and the output of the controller will be smaller.

# Lab Setup

In this lab you will investigate position control of a cart on a linear track using PD controllers. 

The position of the cart can be measured using the linear encoder strip along the edge of the track. An optical encoder counts the pulses, just like in the rotational encoder of Labs 1 an 2. A voltage can be applied to the motor inside the cart, moving the cart along the track using the gear rack. Again, like in labs 1 and 2, the voltage to the motor is applied via an H-bridge. The magnitude of the voltage is set using a PWM signal originating from the Arduino. By increasing the duty cycle of the PWM signal, a larger voltage is applied to the motor. A PD controller has been coded on the Arduino.  

You can set the gains of the PD controller via a Python interface (i.e. this notebook). 

# Learning Objective

In this lab we will be investigating the step response of a closed-loop system governed by a PD controller. Specifically we will investigate how the 2 gains $K_p$, and $K_d$ affect the closed loop response. The key point of this lab is to develop the intuitive understanding of the PD controller. Having the intuition of how the derivative action affects the closed loop system behavior can make tuning PD controllers much easier and faster. In my opinion, that is the main advantage of PID control. Note also that to tune the PD controller you do not need to know a transfer function for the cart!

# Set up

All connections are the same as lab 2, except the encoder. This time, the encoder is plugged in using an 8 pin cable. The white pin on the 8 pin connector is ground. Look on the PCB to ensure the white cable connects to the ground pin. 

![](cable.jpeg)

# The Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ENELX41_lab_util as lu
import control as ct

Set the com port of the Arduino. Refer to the instructions in Labs 1, 2 or 3 if you forget how to determine the correct com port. 

In [ ]:
arduino_com_port = 'COM44'

Some constants that we need

In [ ]:
N = 1000    #data length. Our simulation will be 10 seconds long. 
            #The sampling rate of the system is 100 samples per second

counts_per_inch_to_cm = 1/1200*2.55  # the linear encoder strip has 1200 pulses per inch

The step input. Instead of a step, we will use a square wave, which is just a bunch of steps in series. This will give you a chance to see a few step responses in a row. 

In [ ]:
N = 1000
mag_step = 10
r_step = -mag_step*np.ones(N)                 # a step input of -6cm
r_step[200:400] = mag_step*np.ones(200)       # a step to +6cm
r_step[600:800] = mag_step*np.ones(200)
r_step[800:1000] = np.zeros(200)
r_step_mag, r_step_dir = lu.signal_to_mag_and_dir(r_step)   # the reference needs to be split into a magnitude contained 
                                                            # in 1 byte, and a direction contained in 1 bit.
t = np.linspace(0,10-10/N,N)
fig,ax = plt.subplots(1)
ax.plot(t, r_step)
ax.set_title('Reference Input (Desired Output)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Position (cm)')

# Proportional Action

First we will (again) look at the proportional gain. Slowly increase the proportional gain as 0.01, 0.05, 0.1, 0.25, 0.5. Save your outputs and put them all on the same plot so that you can compare the results. For this experiment, keep $K_i=0$ and $K_d=0$.

Open the Arduino code and implement a proportional controller in lines 162 to 164. Include a screen-shot of the code snippet your wrote for implementing the proportional controller.



# Run the Experiment

When you have finished implementing the controller, and uploading the code to the Arduino, run the experiment using the following code. For all experiments in this lab: please watch your system when you are doing experiments! What is the effect of increasing the proportional gain? 

In [ ]:
Kp = 0.1 #0.05, 0.1, 0.25, 0.5
Kd = 0
#measured_outputs = lu.run_cart_closed_loop_PID(r_step_mag,r_step_dir,Kp,Ki,Kd,com_port=arduino_com_port)

#Kp = 0 # the proportional gain is not used in this part of the lab, but we need to set it to a value.

measured_outputs = lu.run_cart_closed_loop_PD(r_step_mag,r_step_dir,Kp,Kd,com_port=arduino_com_port)

In [ ]:
cart_position_measured = measured_outputs[0]*counts_per_inch_to_cm
motor_input = measured_outputs[1]
dt_measured = measured_outputs[2]
t_measured = np.cumulative_sum(dt_measured)/1000


fig,ax = plt.subplots(3,1, figsize=(6,8))
ax[0].plot(t,r_step)
ax[0].set_title('Reference')
ax[0].set_label('Time (s)')
ax[1].plot(t_measured,motor_input/255*100,label='Motor Input')
ax[1].set_title('Motor Input')
ax[1].set_ylabel('Duty Cycle (percent)')
ax[1].set_xlabel('Time (s)')
ax[2].plot(t_measured,cart_position_measured,label='Measured Cart Position')
ax[2].plot(t,r_step,label='Reference')
ax[2].legend()
ax[2].set_title('Output')
ax[2].set_ylabel('position (cm)')
ax[2].set_xlabel('Time (s)')
fig.tight_layout()


# TA check

Show the TA your plot with all your outputs. 
- What trend do you see? 
- Does the trend make sense?

# Derivative Action

So far your controller has significant overshoot and settling time problems! The solution to reduce overshoot is the add derivative action! Why?

Open the Arduino code and implement a PD controller. The input to the plant should now be:
$$
u(t) = K_p \varepsilon(t) + K_d \frac{d \varepsilon}{dt}
$$
The derivative can be approximated as:
$$
\frac{d \varepsilon}{dt} \approx \varepsilon(t) - \varepsilon(t-T_s)
$$
where $\varepsilon(t-T_s)$ is the error at the previous time step. Note that we don't have to divide by $T_s$ since we are just going to be multiplying the derivative by the gain $K_d$. Also note that this is the naive way of implementing the derivative action (we are not using the modified derivative). This is so that we can observe the effect of sensor noise on the system.

# The Experiment

Slowly increase the derivative gain 0.1, 0.5, 1, and 2. For this experiment set the proportional gain $K_p=0.1$ and $K_i=0$. Save your outputs and put them all on the same plot so that you can compare the results. 


In [ ]:
Kp = 0.1
Kd = 1  #, 0.5, 1, 2, 4, 6
measured_outputs = lu.run_cart_closed_loop_PD(r_step_mag,r_step_dir,Kp,Kd,com_port=arduino_com_port)

In [ ]:
cart_position_measured = measured_outputs[0]*counts_per_inch_to_cm
motor_input = measured_outputs[1]
dt_measured = measured_outputs[2]
t_measured = np.cumulative_sum(dt_measured)/1000


fig,ax = plt.subplots(3,1)
ax[0].plot(t,r_step)
ax[0].set_title('Reference')
ax[1].plot(t_measured,motor_input/255*100,label='Motor Input')
ax[1].set_title('Motor Input')
ax[1].set_ylabel('duty cycle (percent)')
ax[2].plot(t_measured,cart_position_measured,label='Measured Cart Position')
ax[2].plot(t,r_step/255*6*1200*counts_per_inch_to_cm,label='Reference')
ax[2].legend()
ax[2].set_title('Output')
ax[2].set_ylabel('position (cm)')

# Questions
- What trend do you see? 
- Does the trend make sense (Can you explain the trend in terms of what the derivitave action is doing)?
- Do you see noise amplification? What is noise amplification and why does it happen (what happens when you calculate the slope of a a noisy signal)?

# End of Lab